# Create JSON Upgrade

Faster run/lumi JSON creation using `uproot` and file-level thread parallelism. Configure `SAMPLE_PREFIXES`, `MAX_WORKERS`, and `CHECKPOINT_EVERY` in the first code cell before running.

In [1]:
import json
import os
import sys
import time
import traceback
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

import yaml
import uproot
from tqdm.notebook import tqdm

sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path:
    sys.path.insert(1, sidm_path)

from sidm.tools import utilities

yaml_file_path = "/home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/configs/ntuples/data_skimmed.yaml"
output_dir = "/home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON"
log_dir = os.path.join(output_dir, "logs")

os.makedirs(output_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

# Set to None to process all samples. The B-only default avoids rechecking A/C/D.
SAMPLE_PREFIXES = ("DoubleMuon_2018B",)
MAX_WORKERS = 24
CHECKPOINT_EVERY = 250
REPLACE_XCACHE = False

# Retry transient xrootd/uproot failures. Keep worker count modest when retries are active.
MAX_RETRIES = 3
RETRY_SLEEP_SECONDS = 5
RETRY_BACKOFF = 2

with open(yaml_file_path, "r") as f:
    data = yaml.safe_load(f)

all_samples = list(data["llpNanoAOD_v2"]["samples"].keys())
if SAMPLE_PREFIXES is None:
    data_all = all_samples
else:
    data_all = [s for s in all_samples if s.startswith(SAMPLE_PREFIXES)]

print(f"Total samples in YAML: {len(all_samples)}")
print(f"Samples selected: {len(data_all)}")
print(data_all)


Total samples in YAML: 68
Samples selected: 14
['DoubleMuon_2018B_0', 'DoubleMuon_2018B_1', 'DoubleMuon_2018B_2', 'DoubleMuon_2018B_3', 'DoubleMuon_2018B_4', 'DoubleMuon_2018B_5', 'DoubleMuon_2018B_6', 'DoubleMuon_2018B_7', 'DoubleMuon_2018B_8', 'DoubleMuon_2018B_9', 'DoubleMuon_2018B_10', 'DoubleMuon_2018B_11', 'DoubleMuon_2018B_12', 'DoubleMuon_2018B_13']


In [2]:
def write_log_header(f, sample):
    f.write(f"# Sample: {sample}\n")
    f.write(f"# Created: {datetime.now().isoformat()}\n\n")


def build_lumi_dict(file_map):
    lumi_dict = defaultdict(set)
    for pairs in file_map.values():
        for run, lumi in pairs:
            lumi_dict[int(run)].add(int(lumi))
    return lumi_dict


def build_cms_json(lumi_dict):
    cms_json = {}
    for run, lumis in lumi_dict.items():
        lumis = sorted(lumis)
        if len(lumis) == 0:
            continue
        ranges = []
        start = lumis[0]
        prev = lumis[0]
        for lumi in lumis[1:]:
            if lumi == prev + 1:
                prev = lumi
            else:
                ranges.append([start, prev])
                start = lumi
                prev = lumi
        ranges.append([start, prev])
        cms_json[str(run)] = ranges
    return cms_json


def write_outputs(processed_json, filemap_json, cms_json, file_map):
    tmp_processed = processed_json + ".tmp"
    tmp_filemap = filemap_json + ".tmp"
    with open(tmp_processed, "w") as f:
        json.dump(cms_json, f, indent=2)
    with open(tmp_filemap, "w") as f:
        json.dump(file_map, f, indent=2)
    os.replace(tmp_processed, processed_json)
    os.replace(tmp_filemap, filemap_json)


def get_existing_error_counts(error_log):
    counts = {"open": 0, "read": 0}
    if not os.path.exists(error_log):
        return counts
    with open(error_log, "r") as f:
        for line in f:
            if line.startswith("Open errors:"):
                counts["open"] = int(float(line.split(":", 1)[1].strip()))
            elif line.startswith("Read errors:"):
                counts["read"] = int(float(line.split(":", 1)[1].strip()))
    return counts


def load_existing_file_map(filemap_json):
    if not os.path.exists(filemap_json):
        return {}
    with open(filemap_json, "r") as f:
        data = json.load(f)
    return {file_path: [[int(run), int(lumi)] for run, lumi in pairs] for file_path, pairs in data.items()}


def classify_error(error):
    text = str(error).lower()
    if "permission denied" in text:
        return "permission_denied"
    if "operation expired" in text or "timeout" in text or "timed out" in text:
        return "timeout"
    if "no events tree" in text:
        return "missing_events_tree"
    if "failed to close" in text:
        return "close_error"
    if "bytes failed to read" in text:
        return "read_timeout"
    if "unable to open" in text:
        return "unable_to_open"
    return "other"


def is_retryable_error(error):
    return classify_error(error) in {"timeout", "read_timeout", "close_error", "unable_to_open", "other"}


def error_summary(items):
    summary = defaultdict(int)
    for item in items:
        summary[classify_error(item.get("error", ""))] += 1
    return dict(sorted(summary.items()))


def process_file(file_path):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 2):
        root_file = None
        try:
            root_file = uproot.open(file_path)
            if "Events" not in root_file:
                return {"status": "read_error", "file": file_path, "error": "No Events tree", "attempts": attempt}
            tree = root_file["Events"]
            arrays = tree.arrays(["run", "luminosityBlock"], library="np")
            runs = arrays["run"]
            lumis = arrays["luminosityBlock"]
            if len(runs) == 0 or len(lumis) == 0:
                return {"status": "empty", "file": file_path, "attempts": attempt}
            pairs = sorted({(int(run), int(lumi)) for run, lumi in zip(runs, lumis)})
            return {"status": "ok", "file": file_path, "pairs": pairs, "n_events": len(runs), "attempts": attempt}
        except OSError as exc:
            last_error = {"status": "open_error", "file": file_path, "error": str(exc), "attempts": attempt}
        except Exception as exc:
            last_error = {"status": "read_error", "file": file_path, "error": str(exc), "traceback": traceback.format_exc(), "attempts": attempt}
        finally:
            if root_file is not None:
                try:
                    root_file.close()
                except Exception:
                    pass
        if not is_retryable_error(last_error.get("error", "")):
            break`
        if attempt <= MAX_RETRIES:
            time.sleep(RETRY_SLEEP_SECONDS * (RETRY_BACKOFF ** (attempt - 1)))
    return last_error


def write_error_log(error_log, sample, n_files, elapsed, open_errors, read_errors, empty_files, retried_ok):
    with open(error_log, "w") as f:
        write_log_header(f, sample)
        f.write(f"Total files: {n_files}\n")
        f.write(f"Elapsed seconds: {elapsed:.1f}\n")
        f.write(f"Open errors: {len(open_errors)}\n")
        f.write(f"Read errors: {len(read_errors)}\n")
        f.write(f"Empty files: {len(empty_files)}\n")
        f.write(f"Files recovered by retry: {retried_ok}\n")
        f.write(f"Open error summary: {json.dumps(error_summary(open_errors), sort_keys=True)}\n")
        f.write(f"Read error summary: {json.dumps(error_summary(read_errors), sort_keys=True)}\n\n")
        if open_errors:
            f.write("[OPEN_ERRORS]\n")
            for item in open_errors:
                f.write(json.dumps(item) + "\n")
            f.write("\n")
        if read_errors:
            f.write("[READ_ERRORS]\n")
            for item in read_errors:
                f.write(json.dumps(item) + "\n")
            f.write("\n")
        if empty_files:
            f.write("[EMPTY_FILES]\n")
            for file_path in empty_files:
                f.write(file_path + "\n")


In [3]:
for sample in data_all:
    print(r"\\n" + "=" * 80)
    print(f"Processing sample: {sample}")

    processed_json = os.path.join(output_dir, f"processed_lumis_{sample}.json")
    filemap_json = os.path.join(output_dir, f"file_runlumi_map_{sample}.json")
    error_log = os.path.join(log_dir, f"error_log_{sample}.log")

    is_complete = os.path.exists(processed_json) and os.path.exists(filemap_json) and os.path.exists(error_log)
    existing_errors = get_existing_error_counts(error_log)
    if is_complete and existing_errors["open"] == 0 and existing_errors["read"] == 0:
        print("  -> Skip: outputs exist and open/read error counts are zero")
        print(f"     {processed_json}")
        print(f"     {filemap_json}")
        print(f"     {error_log}")
        continue
    if is_complete:
        print(f"  -> Existing outputs have errors; retrying missing files only: {existing_errors}")

    try:
        fileset_data = utilities.make_fileset(
            [sample],
            "llpNanoAOD_v2",
            max_files=-1,
            location_cfg="data_skimmed.yaml",
            replace_xcache=REPLACE_XCACHE,
        )
    except Exception as exc:
        print(f"  -> Error making fileset for sample {sample}")
        print(f"     {exc}")
        with open(error_log, "w") as f:
            write_log_header(f, sample)
            f.write("[FILESET_ERROR]\n")
            f.write(str(exc) + "\n")
        continue

    file_list = fileset_data.get(sample, {}).get("files", [])
    n_files = len(file_list)
    print(f"  -> Number of files: {n_files}")
    print(f"  -> Workers: {MAX_WORKERS}")
    print(f"  -> Retries per file: {MAX_RETRIES}")

    if n_files == 0:
        print("  -> Empty file list, skip")
        with open(error_log, "w") as f:
            write_log_header(f, sample)
            f.write("[STATUS]\n")
            f.write("File list is empty.\n")
        continue

    file_map = load_existing_file_map(filemap_json) if is_complete else {}
    pending_files = [file_path for file_path in file_list if file_path not in file_map]
    print(f"  -> Existing mapped files: {len(file_map)}")
    print(f"  -> Pending files: {len(pending_files)}")

    open_errors = []
    read_errors = []
    empty_files = []
    retried_ok = 0
    start_time = time.time()

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(process_file, file_path) for file_path in pending_files]
        for done_count, future in enumerate(tqdm(as_completed(futures), total=len(futures)), start=1):
            result = future.result()
            status = result["status"]
            file_path = result["file"]
            attempts = result.get("attempts", 1)
            if status == "ok":
                if attempts > 1:
                    retried_ok += 1
                pairs = result["pairs"]
                file_map[file_path] = [[run, lumi] for run, lumi in pairs]
            elif status == "empty":
                empty_files.append(file_path)
            elif status == "open_error":
                open_errors.append({"file": file_path, "error": result.get("error", ""), "attempts": attempts})
            else:
                read_errors.append({"file": file_path, "error": result.get("error", ""), "traceback": result.get("traceback", ""), "attempts": attempts})
            if CHECKPOINT_EVERY and done_count % CHECKPOINT_EVERY == 0:
                cms_json = build_cms_json(build_lumi_dict(file_map))
                write_outputs(processed_json, filemap_json, cms_json, file_map)
                print(f"  -> Checkpoint saved after {done_count}/{len(pending_files)} pending files")

    cms_json = build_cms_json(build_lumi_dict(file_map))
    write_outputs(processed_json, filemap_json, cms_json, file_map)
    elapsed = time.time() - start_time
    write_error_log(error_log, sample, n_files, elapsed, open_errors, read_errors, empty_files, retried_ok)

    print(f"  -> Done in {elapsed:.1f} s")
    print(f"  -> Runs collected: {len(cms_json)}")
    print(f"  -> Files mapped: {len(file_map)}")
    print(f"  -> Files recovered by retry: {retried_ok}")
    print(f"  -> Open errors: {len(open_errors)}")
    print(f"  -> Read errors: {len(read_errors)}")
    print(f"  -> Empty files: {len(empty_files)}")
    print(f"     {processed_json}")
    print(f"     {filemap_json}")
    print(f"     {error_log}")


\\n================================================================================
Processing sample: DoubleMuon_2018B_0
  -> Skip: outputs exist and open/read error counts are zero
     /home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/processed_lumis_DoubleMuon_2018B_0.json
     /home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/file_runlumi_map_DoubleMuon_2018B_0.json
     /home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/logs/error_log_DoubleMuon_2018B_0.log
\\n================================================================================
Processing sample: DoubleMuon_2018B_1
  -> Skip: outputs exist and open/read error counts are zero
     /home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/processed_lumis_DoubleMuon_2018B_1.json
     /home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/file_runlumi_map_DoubleMuon_2018B_1.json
     /home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/logs/error_log_DoubleMuon_2018B_1.log
\\n=====================

  0%|          | 0/7344 [00:00<?, ?it/s]

  -> Checkpoint saved after 250/7344 pending files
  -> Checkpoint saved after 500/7344 pending files
  -> Checkpoint saved after 750/7344 pending files
  -> Checkpoint saved after 1000/7344 pending files
  -> Checkpoint saved after 1250/7344 pending files
  -> Checkpoint saved after 1500/7344 pending files
  -> Checkpoint saved after 1750/7344 pending files
  -> Checkpoint saved after 2000/7344 pending files
  -> Checkpoint saved after 2250/7344 pending files
  -> Checkpoint saved after 2500/7344 pending files
  -> Checkpoint saved after 2750/7344 pending files
  -> Checkpoint saved after 3000/7344 pending files
  -> Checkpoint saved after 3250/7344 pending files
  -> Checkpoint saved after 3500/7344 pending files
  -> Checkpoint saved after 3750/7344 pending files
  -> Checkpoint saved after 4000/7344 pending files
  -> Checkpoint saved after 4250/7344 pending files
  -> Checkpoint saved after 4500/7344 pending files
  -> Checkpoint saved after 4750/7344 pending files
  -> Checkpoint

  0%|          | 0/9386 [00:00<?, ?it/s]

  -> Checkpoint saved after 250/9386 pending files
  -> Checkpoint saved after 500/9386 pending files
  -> Checkpoint saved after 750/9386 pending files
  -> Checkpoint saved after 1000/9386 pending files
  -> Checkpoint saved after 1250/9386 pending files
  -> Checkpoint saved after 1500/9386 pending files
  -> Checkpoint saved after 1750/9386 pending files
  -> Checkpoint saved after 2000/9386 pending files
  -> Checkpoint saved after 2250/9386 pending files
  -> Checkpoint saved after 2500/9386 pending files
  -> Checkpoint saved after 2750/9386 pending files
  -> Checkpoint saved after 3000/9386 pending files
  -> Checkpoint saved after 3250/9386 pending files
  -> Checkpoint saved after 3500/9386 pending files
  -> Checkpoint saved after 3750/9386 pending files
  -> Checkpoint saved after 4000/9386 pending files
  -> Checkpoint saved after 4250/9386 pending files
  -> Checkpoint saved after 4500/9386 pending files
  -> Checkpoint saved after 4750/9386 pending files
  -> Checkpoint

...

  -> Done in 24.8 s
  -> Runs collected: 1
  -> Files mapped: 25
  -> Files recovered by retry: 0
  -> Open errors: 0
  -> Read errors: 0
  -> Empty files: 0
     /home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/processed_lumis_DoubleMuon_2018B_11.json
     /home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/file_runlumi_map_DoubleMuon_2018B_11.json
     /home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/logs/error_log_DoubleMuon_2018B_11.log
\\n================================================================================
Processing sample: DoubleMuon_2018B_12
  -> Number of files: 31
  -> Workers: 24
  -> Retries per file: 3
  -> Existing mapped files: 0
  -> Pending files: 31


  0%|          | 0/31 [00:00<?, ?it/s]

  -> Done in 28.8 s
  -> Runs collected: 2
  -> Files mapped: 31
  -> Files recovered by retry: 0
  -> Open errors: 0
  -> Read errors: 0
  -> Empty files: 0
     /home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/processed_lumis_DoubleMuon_2018B_12.json
     /home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/file_runlumi_map_DoubleMuon_2018B_12.json
     /home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/logs/error_log_DoubleMuon_2018B_12.log
\\n================================================================================
Processing sample: DoubleMuon_2018B_13
  -> Number of files: 29
  -> Workers: 24
  -> Retries per file: 3
  -> Existing mapped files: 0
  -> Pending files: 29


  0%|          | 0/29 [00:00<?, ?it/s]

  -> Done in 27.4 s
  -> Runs collected: 1
  -> Files mapped: 29
  -> Files recovered by retry: 0
  -> Open errors: 0
  -> Read errors: 0
  -> Empty files: 0
     /home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/processed_lumis_DoubleMuon_2018B_13.json
     /home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/file_runlumi_map_DoubleMuon_2018B_13.json
     /home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/logs/error_log_DoubleMuon_2018B_13.log
